# Statistical Audit of pandas-dev/pandas



## Member Information
- **Name:** Ahmad Aqil Fadria
- **Role:** Data Engineer

---

# Research Questions

1. Berapa perkiraan probabilitas bahwa sebuah pull request akan digabungkan ke pandas-dev/pandas?

2. Apakah rata-rata durasi penutupan issue telah berubah secara signifikan setelah rilis utama pandas?

3. Berapa probabilitas bahwa issue yang dipilih secara acak membutuhkan waktu lebih dari 30 hari untuk ditutup?

---

# Import Libraries

In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# GitHub API Configuration

Pada bagian ini dilakukan konfigurasi repository GitHub
dan endpoint API yang digunakan untuk mengambil data
issue dan pull request dari repository pandas-dev/pandas.

Konfigurasi ini digunakan sebagai dasar proses pengambilan data
menggunakan GitHub REST API sehingga notebook dapat
mengakses data repository secara otomatis.


In [ ]:
OWNER = "pandas-dev"
REPO = "pandas"

BASE_URL = f"https://api.github.com/repos/{OWNER}/{REPO}/issues"

# GitHub Personal Access Token
TOKEN = "ghp_xxxxxxxxxxxxxxxxxxxxx"

headers = {
    "Authorization": f"token {TOKEN}",
    "Accept": "application/vnd.github+json"
}


# Collect Issues Data

Pada bagian ini dilakukan proses pengambilan data issue
dari GitHub REST API pada repository pandas-dev/pandas.

Data yang dikumpulkan mencakup issue dengan status
open maupun closed yang nantinya akan digunakan
untuk proses cleaning, exploratory data analysis (EDA),
dan analisis statistik pada tahap berikutnya.


In [ ]:
all_issues = []

for page in range(1, 51):

    url = (
        f"{BASE_URL}"
        f"?state=all&per_page=100&page={page}"
    )

    response = requests.get(
        url,
        headers=headers
    )


    if response.status_code != 200:
        print(f"Error collecting page {page}")
        break

    data = response.json()

    if len(data) == 0:
        break

    all_issues.extend(data)

    
    print("Total raw data:",
      len(all_issues))

# Convert to DataFrame

Data hasil pengambilan dari GitHub API masih berbentuk
JSON sehingga perlu dikonversi menjadi pandas DataFrame.

Proses ini dilakukan agar data lebih mudah dibaca,
dibersihkan, dianalisis, dan divisualisasikan
pada tahap selanjutnya.


In [ ]:
df = pd.DataFrame(all_issues)

print(df.shape)

# Save Raw Dataset

Dataset asli hasil pengambilan dari GitHub API
disimpan tanpa perubahan ke dalam folder raw.

Penyimpanan data mentah ini bertujuan untuk menjaga
keaslian data sebelum dilakukan proses cleaning
dan preprocessing lebih lanjut.


In [ ]:
df.to_csv("../data/raw/issues_raw.csv", index=False)

print("Raw dataset saved.")

# Filter True Issues

Endpoint issues pada GitHub juga dapat mengandung
data pull request sehingga perlu dilakukan proses filtering.

Pada bagian ini dilakukan pemisahan data agar
dataset hanya berisi issue yang sebenarnya
dan tidak mencampurkan data pull request.


In [ ]:
# ============================================================
# CLEAN ISSUE DATA
# ============================================================

issues_df = df.copy()

# ambil hanya closed
issues_df = issues_df[
    issues_df["state"] == "closed"
].copy()

print(issues_df.shape)

# Select Important Columns

Pada bagian ini dipilih beberapa kolom penting
yang relevan untuk kebutuhan analisis statistik.

Pemilihan kolom dilakukan agar dataset menjadi
lebih ringkas, mudah diproses, dan fokus
pada variabel yang akan dianalisis.


In [ ]:
issues_df = issues_df[
    [
        "id",
        "number",
        "title",
        "state",
        "created_at",
        "closed_at",
        "comments"
    ]
]

# Datetime Conversion

Kolom yang berisi informasi waktu dikonversi
ke format datetime pada pandas.

Proses ini dilakukan agar data waktu dapat
digunakan untuk analisis durasi, perhitungan selisih waktu,
serta kebutuhan analisis statistik lainnya.


In [ ]:
issues_df["created_at"] = pd.to_datetime(
    issues_df["created_at"]
)

issues_df["closed_at"] = pd.to_datetime(
    issues_df["closed_at"]
)

# Feature Engineering

Pada bagian ini dilakukan pembuatan variabel baru
yang dibutuhkan untuk analisis statistik.

Variabel tambahan dibuat berdasarkan data yang sudah ada
agar proses analisis, visualisasi, dan interpretasi data
dapat dilakukan dengan lebih mudah.


In [ ]:
issues_df["close_duration_days"] = (
    issues_df["closed_at"] - issues_df["created_at"]
).dt.days

issues_df["is_closed"] = 1

# Save Clean Dataset

Dataset yang telah melalui proses cleaning
dan preprocessing disimpan ke dalam folder clean.

Dataset bersih ini nantinya akan digunakan
pada tahap analisis statistik berikutnya
seperti estimation, inference, hypothesis testing,
dan simulation.


In [ ]:
issues_df.to_csv(
    "../data/clean/dataset.csv",
    index=False
)

# Dataset Overview

Pada bagian ini ditampilkan gambaran umum
mengenai dataset yang telah dibersihkan.

Overview dataset digunakan untuk melihat
struktur data, jumlah data, tipe variabel,
serta memastikan data siap digunakan
untuk proses analisis selanjutnya.


In [ ]:
issues_df.head()

In [ ]:
issues_df.info()

In [ ]:
issues_df.describe()

# Missing Values Analysis

Pada bagian ini dilakukan pengecekan
terhadap missing values pada dataset.

Analisis ini bertujuan untuk memastikan
tidak terdapat data yang kosong atau tidak lengkap
yang dapat memengaruhi hasil analisis statistik.


In [ ]:
issues_df.isnull().sum()

# Visualization: Closing Duration Distribution

Visualisasi histogram ini menunjukkan distribusi
durasi waktu penutupan issue dalam satuan hari.

Berdasarkan histogram yang dihasilkan,
distribusi data cenderung skewed ke kanan (right-skewed),
yang menunjukkan bahwa sebagian besar issue
ditutup dalam waktu relatif singkat,
namun terdapat beberapa issue yang membutuhkan
waktu jauh lebih lama untuk diselesaikan.

Pola distribusi seperti ini umum ditemukan
pada data repository open-source karena
tidak semua issue memiliki tingkat kesulitan yang sama.


In [ ]:
plt.figure(figsize=(10, 5))

sns.histplot(
    issues_df["close_duration_days"],
    bins=30
)

plt.title("Distribution of Issue Closing Duration")
plt.xlabel("Days")
plt.ylabel("Frequency")

plt.show()

# Visualization: Issue Status Count

Visualisasi ini menunjukkan perbandingan jumlah
issue berdasarkan statusnya, yaitu open dan closed.

Berdasarkan hasil visualisasi, jumlah issue dengan status
closed lebih banyak dibandingkan issue dengan status open.
Hal ini menunjukkan bahwa sebagian besar issue pada
repository pandas-dev/pandas telah berhasil diselesaikan.

Perbandingan jumlah issue open dan closed dapat memberikan
gambaran mengenai tingkat aktivitas dan efektivitas
pengelolaan issue dalam repository open-source.



In [ ]:
plt.figure(figsize=(6, 4))

ax = df["state"].value_counts().plot(
    kind="bar",
    color=["blue", "orange"]
)

# menampilkan jumlah di atas batang
for i in ax.patches:
    ax.text(
        i.get_x() + i.get_width() / 2,
        i.get_height() + 5,
        str(int(i.get_height())),
        ha='center'
    )

plt.title("Issue Status Count")
plt.xlabel("State")
plt.ylabel("Count")

plt.show()

# Visualization: Issues Created Over Time

Visualisasi line chart ini menunjukkan tren jumlah
issue yang dibuat dari waktu ke waktu berdasarkan periode bulan.

Data created_at dikonversi menjadi format year-month
agar jumlah issue dapat dikelompokkan per bulan
dan divisualisasikan dalam bentuk time series.

Melalui grafik ini dapat diamati pola aktivitas repository,
seperti peningkatan maupun penurunan jumlah issue
pada periode tertentu.

Fluktuasi jumlah issue dapat menunjukkan:

* tingkat aktivitas pengguna repository
* peningkatan penggunaan project
* adanya update atau perubahan besar pada project
* peningkatan laporan bug atau request fitur baru

Analisis tren waktu seperti ini membantu memahami
perkembangan aktivitas repository secara keseluruhan.
ualization shows issue activity trends over time.

In [ ]:
issues_df["year_month"] = (
    issues_df["created_at"]
    .dt.to_period("M")
    .astype(str)
)

monthly_issues = (
    issues_df["year_month"]
    .value_counts()
    .sort_index()
)

plt.figure(figsize=(12, 5))

monthly_issues.plot()

plt.title("Issues Created Over Time")
plt.xlabel("Year-Month")
plt.ylabel("Number of Issues")

plt.xticks(rotation=45)

plt.show()

# Variable Selection

Pada bagian ini dipilih beberapa variabel
yang akan digunakan untuk proses analisis statistik
pada tahap selanjutnya.

Pemilihan variabel dilakukan berdasarkan
relevansi data terhadap research question
dan kebutuhan analisis seperti estimation,
inference, hypothesis testing, dan simulation.


In [ ]:
variable_selection = pd.DataFrame({
    "Variable": [
        "is_closed",
        "close_duration_days",
        "comments"
    ],

    "Description": [
        "Status issue closed",
        "Durasi penutupan issue",
        "Jumlah komentar issue"
    ],

    "Statistical Usage": [
        "Bernoulli Estimation",
        "Confidence Interval & Hypothesis Testing",
        "Poisson Analysis"
    ]
})

variable_selection

# Pull Request Data Collection

Pada bagian ini dilakukan proses pengambilan
data pull request dari repository pandas-dev/pandas
menggunakan GitHub REST API.

Dataset pull request ini akan digunakan
untuk analisis statistik pada tahap berikutnya,
seperti:

* estimasi probabilitas pull request di-merge
* Bernoulli Maximum Likelihood Estimation (MLE)
* confidence interval
* hypothesis testing

Data yang dikumpulkan mencakup informasi
status pull request, waktu pembuatan,
waktu merge, dan waktu penutupan pull request.


In [ ]:
# ============================================================
# PULL REQUEST DATA COLLECTION
# ============================================================

all_prs = []

# endpoint pull requests
PR_URL = (
    f"https://api.github.com/repos/"
    f"{OWNER}/{REPO}/pulls"
)

# mengambil beberapa halaman data
for page in range(1, 51):

    url = (
        f"{PR_URL}"
        f"?state=all&per_page=100&page={page}"
    )

    response = requests.get(
        url,
        headers=headers
    )

    # cek request berhasil
    if response.status_code != 200:
        print(f"Error collecting page {page}")
        print(response.text)
        break

    data = response.json()

    # berhenti jika data habis
    if len(data) == 0:
        break

    all_prs.extend(data)

    print(f"Collected PR page {page}")

print("Total Pull Requests:",
      len(all_prs))

# Convert Pull Request Data to DataFrame

Pada bagian ini, data pull request yang telah dikumpulkan dari GitHub REST API dalam format JSON akan dikonversi menjadi DataFrame menggunakan pandas.

Proses ini bertujuan untuk memudahkan pengolahan data, seperti filtering, cleaning, dan analisis statistik pada tahap selanjutnya.

Setelah dikonversi, data dapat dianalisis menggunakan berbagai metode seperti agregasi, visualisasi, dan perhitungan metrik statistik.


In [ ]:
pr_df = pd.DataFrame(all_prs)

print(pr_df.shape)

pr_df.head()

# Save Raw Pull Request Dataset

Pada bagian ini, dataset pull request yang telah dikumpulkan disimpan dalam bentuk file CSV tanpa dilakukan perubahan atau preprocessing.

Tujuan penyimpanan ini adalah untuk menjaga data asli (raw data) dari GitHub REST API agar dapat digunakan kembali jika diperlukan proses analisis ulang.

Output dari proses ini disimpan pada:

* `data/raw/pull_requests_raw.csv`


In [ ]:
pr_df.to_csv(
    "../data/raw/pull_requests_raw.csv",
    index=False
)

print("pull_requests_raw.csv saved.")

# Pull Request Data Cleaning

Pada bagian ini dilakukan proses pembersihan dataset pull request agar siap digunakan untuk analisis statistik.

Proses cleaning meliputi:

* pemilihan kolom yang relevan untuk analisis
* konversi data waktu (datetime conversion)
* penanganan nilai yang hilang (missing values)
* pembuatan fitur baru (feature engineering)

Setelah proses ini, dataset menjadi lebih terstruktur dan siap digunakan untuk analisis lanjutan seperti estimasi, visualisasi, dan pengujian statistik.


In [ ]:
pr_clean = pr_df[
    [
        "id",
        "number",
        "state",
        "created_at",
        "closed_at",
        "merged_at"
    ]
].copy()

print(pr_clean.shape)

pr_clean.head()

# Datetime Conversion

Pada bagian ini, variabel yang berupa timestamp diubah ke dalam format datetime menggunakan pandas.

Proses ini diperlukan agar data waktu dapat dianalisis secara lebih akurat, terutama untuk perhitungan durasi, seperti waktu penyelesaian issue atau pull request.

Dengan format datetime, data dapat digunakan untuk analisis statistik seperti:

* perhitungan selisih waktu (time delta)
* analisis tren waktu
* pengelompokan berdasarkan periode (hari, bulan, tahun)


In [ ]:
pr_clean["created_at"] = pd.to_datetime(
    pr_clean["created_at"]
)

pr_clean["closed_at"] = pd.to_datetime(
    pr_clean["closed_at"]
)

pr_clean["merged_at"] = pd.to_datetime(
    pr_clean["merged_at"]
)

# Pull Request Feature Engineering

Pada bagian ini dilakukan proses pembuatan variabel tambahan (feature engineering) untuk mendukung analisis statistik pada dataset pull request.

Variabel yang dibuat meliputi:

* **merged**: indikator apakah pull request berhasil di-merge atau tidak (bernilai 1 jika merged, 0 jika tidak)
* **review_duration_days**: durasi waktu review atau penyelesaian pull request dalam satuan hari

Variabel ini digunakan untuk analisis lebih lanjut seperti estimasi probabilitas merge, distribusi durasi, serta pengujian statistik.


In [ ]:
# merged PR indicator
pr_clean["merged"] = (
    pr_clean["merged_at"].notna()
).astype(int)

# review duration
pr_clean["review_duration_days"] = (
    pr_clean["closed_at"] - pr_clean["created_at"]
).dt.days

pr_clean.head()

# Save Clean Pull Request Dataset

Pada bagian ini, dataset pull request yang telah melalui proses cleaning dan feature engineering disimpan ke dalam file CSV.

Penyimpanan ini bertujuan untuk menyediakan dataset yang sudah siap digunakan pada tahap analisis statistik selanjutnya.

Output dari proses ini disimpan pada:

* `data/clean/pr_dataset.csv`


In [ ]:
pr_clean.to_csv(
    "../data/clean/pr_dataset.csv",
    index=False
)

print("pr_dataset.csv saved.")

# Visualization: Pull Request Merge Status

Visualisasi ini menunjukkan perbandingan jumlah pull request yang berhasil di-merge dan yang tidak di-merge dalam repository.

Hasil visualisasi ini digunakan untuk memahami distribusi status pull request, yang menjadi dasar dalam analisis statistik.

Variabel ini penting karena digunakan untuk:

* estimasi Bernoulli (merged vs not merged)
* analisis probabilitas merge
* konstruksi confidence interval


In [ ]:
plt.figure(figsize=(6,4))

ax = sns.countplot(
    x="merged",
    data=pr_clean,
    palette=["red", "green"]
)

for container in ax.containers:
    ax.bar_label(container)

plt.title("Pull Request Merge Status")
plt.xlabel("Merged")
plt.ylabel("Count")

plt.show()

## Penjelasan Visualization: PR Review Duration

Distribusi durasi review pull request menunjukkan pola **right-skewed (skewed ke kanan)**.

Artinya sebagian besar pull request memiliki durasi review yang cepat, tetapi terdapat beberapa pull request dengan durasi sangat lama yang menyebabkan ekor distribusi memanjang ke kanan.

Pada kondisi ini, nilai mean cenderung lebih besar dibanding median karena dipengaruhi oleh nilai ekstrem (outlier).


In [ ]:
plt.figure(figsize=(8,5))

sns.histplot(
    pr_clean["review_duration_days"].dropna(),
    bins=20
)

plt.title("PR Review Duration Distribution")
plt.xlabel("Days")
plt.ylabel("Frequency")

plt.show()

## Visualization: Pull Request Activity Over Time

Visualisasi ini menunjukkan jumlah pull request yang dibuat dari waktu ke waktu berdasarkan periode bulanan.

Grafik ini digunakan untuk melihat pola aktivitas kontribusi pada repository, termasuk periode ketika pengembangan sedang aktif atau melambat.

Dengan analisis ini, kita dapat memahami tren perkembangan proyek serta dinamika kontribusi developer dari waktu ke waktu.


In [ ]:
pr_clean["year_month"] = (
    pr_clean["created_at"]
    .dt.to_period("M")
    .astype(str)
)

monthly_pr = (
    pr_clean["year_month"]
    .value_counts()
    .sort_index()
)

plt.figure(figsize=(12,5))

monthly_pr.plot()

plt.title("Pull Requests Created Over Time")
plt.xlabel("Year-Month")
plt.ylabel("Number of PRs")

plt.xticks(rotation=45)

plt.show()

## Ringkasan

Notebook ini berhasil menyelesaikan proses data engineering untuk statistical audit pada repository `pandas-dev/pandas`.

Proses yang dilakukan meliputi:

* pengumpulan data issue dan pull request dari GitHub API
* penyimpanan dataset raw
* data cleaning dan preprocessing
* feature engineering
* exploratory data analysis (EDA)
* visualisasi statistik

Dataset yang dihasilkan:

* Raw: `data/raw/issues_raw.csv`, `data/raw/pull_requests_raw.csv`
* Clean: `data/clean/dataset.csv`, `data/clean/pr_dataset.csv`

Variabel utama yang digunakan untuk analisis lanjutan:

* durasi penutupan issue
* frekuensi aktivitas issue
* status merge pull request
* durasi review pull request

Dataset ini akan digunakan pada tahap berikutnya untuk:

* estimasi parameter
* confidence interval
* hypothesis testing
* simulasi komputasi

Notebook ini menjadi fondasi utama untuk seluruh analisis statistik pada project ini.
